### Replicação do experimento de Alfarham 

Código do Alfarham disponível em: https://github.com/DeepWave-KAUST/DeepFWIHessian/tree/main/notebooks/Exp_Marmousi_dm

Artigo do Alfarham disponível em: https://doi.org/10.1093/gji/ggae378

Objetivo:



### Ficha Técnica de Replicação: Modelo Alfarhan (Marmousi)

| Categoria | Parâmetro / Detalhe | Valor no Artigo |
| :--- | :--- | :--- |
| **Arquitetura** | **Tipo de Rede** | U-Net (n_channels=1, n_classes=1, hidden_channels=128)|
| **Arquitetura** | **Função de Ativação** |Leaky ReLU (α=0.2) |
| **Dados** | **Tamanho do Grid** | 221 (profundidade/nz) × 601 (largura/nx) |
| **Treino** | **Função de Perda (Loss)**  | Erro Médio Quadrático / MSE|
| **Treino** | **Otimizador** |Adam |
| **Treino** | **Learning Rate** (Valor inicial e se há scheduler) | $10^{−4}$, com scheduler ativado |
| **Performance** | **Convergência inicial** (1000 épocas) |1.041 s (~17,3 min) |
| **Performance** | **Regime permanente** (300 épocas) |329 s (~5,5 min) |.

In [2]:
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')
import gc
import sys
import os
import time
import datetime
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import scipy as sp
import cv2
from tqdm.notebook import tqdm
from skimage.metrics import structural_similarity as ssim
import deepwave

In [ ]:
# Modifique o caminho abaixo se a pasta 'deepinvhessian' estiver em outro lugar
sys.path.append(os.path.abspath("../"))  # Volta uma pasta (se o notebook estiver em proj-02/notebooks)
# Se não funcionar, tente: sys.path.append(os.path.abspath("../../"))

from deepinvhessian import fwi
from deepinvhessian.utilities import *
from deepinvhessian.filters import *
from deepinvhessian.train import *
from deepinvhessian.masks import *
from unet import *

In [ ]:
set_seed(14)
# setting device on GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

#Additional Info when using cuda
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))
    print('Memory Usage:')
    print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
    print('Cached:   ', round(torch.cuda.memory_reserved(0)/1024**3,1), 'GB')

In [ ]:
# Define the model and acquisition parameters
par = {'nx':601,   'dx':0.015, 'ox':0,
       'nz':221,   'dz':0.015, 'oz':0,
       'num_shots':30,    'ds':0.3,   'os':0,  'sz':0,
       'num_receivers_per_shot':300,   'dr':0.03,  'orec':0, 'rz':0,
       'nt':5000,  'dt':0.001,  'ot':0,
       'freq': 5, 'num_sources_per_shot':1, 'num_dims':2,
       'num_batches':30,
        'FWI_itr': 100
      }

velocity_file = 'data/Marm.bin'

In [ ]:
# Load the true model
model_true = (np.fromfile(velocity_file, np.float32)
              .reshape(par['nz'], par['nx']))

# function to get water layer mask
def mask(model,value):
    """
    Return a mask for the model (m) using the (value)
    """
    mask = model > value
    mask = mask.astype(int)
    mask[:21] = 0
    return mask

mask = mask(model_true, 1.5)

In [ ]:
m_vmin, m_vmax = np.percentile(model_true, [2,98])
show_model(model_true, cmap='jet', vmin=m_vmin, vmax=m_vmax, figsize=(12, 5), extent=(0, par['nx']*par['dx']*1000, par['nz']*par['dx']*1000, 0),
           title='Marmousi true model')

In [ ]:
# Create initial guess model for inversion by smoothing the true model
model_init = sp.ndimage.gaussian_filter(model_true, sigma=[10,15])
model_init = model_init * mask
model_init[model_init==0] = 1.5 # km/s
model_init = model_init.astype(np.float32)
show_model(model_init, cmap='jet', vmin=m_vmin, vmax=m_vmax, figsize=(12, 5), extent=(0, par['nx']*par['dx']*1000, par['nz']*par['dx']*1000, 0),
           title='Marmousi initial model')

In [ ]:
# Create the source the wavelet
source_wavelet = deepwave.wavelets.ricker(par['freq'], par['nt'], par['dt'], 1/par['freq'])
# Initialize the FWI class
params = fwi.FWIParams(par, torch.tensor(source_wavelet), 1)
# Get the source receiver coordinates
x_s1, x_r1 = params.get_coordinate(1)
# Create a wavelet for every source
source_amplitudes = params.create_wavelet(torch.tensor(source_wavelet))

In [ ]:
# Força a conversão das coordenadas espaciais para índices inteiros absolutos da malha
params.s_cor = params.s_cor.round().long()
params.r_cor = params.r_cor.round().long()

In [ ]:
# Visualize the source wavelet
plt.plot(np.arange(0,par['nt'])*par['dt'], source_amplitudes[0,0,:])
plt.xlabel('Time (s)')
plt.title('Source wavelet')

In [ ]:
# Simulate the true data
data_true = fwi.forward_modelling(params, torch.tensor(model_true).float(), device)

In [ ]:
# 1. Elimine os residuais decimais forçando o cast para índices inteiros
x_s1 = x_s1.round().long()
x_r1 = x_r1.round().long()

# Precisei editar a função source_ilumination em FWI para rodar
# Compute source illumination
SI = fwi.source_illumination(torch.tensor(model_init), source_amplitudes, par['dx'], par['dt'], x_s1, device=device)
# clear memory
torch.cuda.empty_cache()
gc.collect()
# Visualize the source illumination
simin, simax = np.percentile(SI.cpu(), [2,98])
show_model(SI.cpu(), cmap='bwr', vmin=simin, vmax=simax, figsize=(12, 5), extent=(0, par['nx']*par['dx']*1000, par['nz']*par['dx']*1000, 0),
           title='Source Illumination')

In [ ]:
# Create folder to save the results
exp_name = f'Exp_Marmousi_dm'
if os.path.isdir(exp_name) is False:
    os.makedirs(exp_name)

In [ ]:
# Run FWI with the proposed method

# Move data to GPU if using GPU
model = torch.tensor(model_init).clone().to(device)
model.requires_grad = True
data_true = torch.tensor(data_true).float()
mask = torch.tensor(mask).to(device)
# Create lists to save results
gradients, dm1s, gradients_pred, dms, updates, fwi_loss, ssim_list, network_loss = [], [], [], [], [], [], [], []

data_range = model_true.max() - model_true.min()
loss_fn = torch.nn.MSELoss() # Misfit function for FWI and Born modelling
optimizer = torch.optim.SGD([{'params': [model], 'lr': 1e-2}]) # Optimizer to run FWI with step size: lr
# Create the network, its optimizer and the loss function to train it
network = UNet(n_channels=1, n_classes=1, hidden_channels=128).to(device)
optimizer_unet = torch.optim.Adam(network.parameters(), lr=1e-4)
l2_norm = torch.nn.MSELoss()
network_iter_init = 1000 # Number of epochs to train the network in the first FWI iteration
network_iter_fin = 300 # Number of epochs to train the network in every FWI iteration except the first one
tsamples = 0 # Number of time samples starting from zero to exclude from computing the misfit
FWI_iter = 2 # Number of FWI iterations
t_start = time.time()
for iteration in tqdm(range(FWI_iter)):
    # Compute the structural similarity index measure (ssim) between the current and the true models
    ssim_metric = ssim(model.detach().cpu().numpy(), model_true, data_range=data_range)
    ssim_list.append(ssim_metric)
    # Compute FWI gradient
    optimizer.zero_grad()
    grad, iter_loss = fwi.compute_gradient(params, model, data_true, loss_fn, tsamples, device)
    fwi_loss.append(iter_loss)
    print(f'FWI iteration: {iteration} loss = {fwi_loss[-1]}, ssim = {ssim_list[-1]}')
    # Clip the gradient values
    torch.nn.utils.clip_grad_value_(model, torch.quantile(grad.detach().abs(), 0.98))
    # Apply source illumination to the gradient
    grad = (grad * model.detach().clone()**3 ) / SI
    if iteration == 0: gmax0 =  torch.abs(grad.detach()).max()
    # Normalize the gradient, mask it around the sources and apply taperinn to the shallower and deeper parts
    grad = (grad /gmax0) * mask
    gradients.append(grad.cpu().detach().numpy())
    # Compute dm1 with the gradient as the perturbation
    dm1 = grad.detach().clone().to(device)
    dm1.requires_grad = True
    dm1 = fwi.compute_dm1(params, model.detach().clone(), dm1 , loss_fn, tsamples, device)
    # Apply source illumination to dm1
    dm1 = dm1 * (model.detach().clone() ** 3) / SI
    if iteration == 0: dm1max0 =  1e1 * torch.abs(dm1.detach()).max()
    # Normalize dm1 and mask it around the sources
    dm1 = (dm1 / dm1max0)  * mask
    dm1s.append(dm1.cpu().detach().numpy())
    # Train the network
    training_pair = {'x': dm1.clone().unsqueeze(0).unsqueeze(0),
                    'y': grad.clone().unsqueeze(0).unsqueeze(0)}
    network_iter = network_iter_init if iteration == 0 else network_iter_fin
    lossn = train(network, training_pair, optimizer_unet, l2_norm, network_iter, use_scheduler=True, device=device)
    network_loss.extend(lossn)
    # Get the gradient from the network
    with torch.no_grad():
        g = network(training_pair['x']).squeeze() * mask
    gradients_pred.append(g.cpu().detach().numpy())
    # Get dm from the network
    with torch.no_grad():
        dm = network(training_pair['y']).squeeze() * mask
    dms.append(dm.cpu().detach().numpy())
    # Update the model
    model.grad.data[:] = dm.detach().clone()
    optimizer.step()
    updates.append(model.detach().clone().cpu().numpy())
    # Plot the results
    # show_one_iter_dm(grad.cpu(), dm1.cpu(), g.cpu(), dm.cpu(), model.detach().cpu(), lossn, iteration=iteration,
    #             cmap='bwr', vmin=m_vmin, vmax=m_vmax, extent=(0, par['nx']*par['dx']*1000, par['nz']*par['dx']*1000, 0), save_path=f'{exp_name}')
t_end = time.time()
t_delta = t_end - t_start
print(f'Runtime:{datetime.timedelta(seconds=t_delta)}')
# Save the results
np.savez(f'{exp_name}/losses', fwi_loss=np.array(fwi_loss),
                               network_loss=np.array(network_loss),
                               ssim=np.array(ssim_list),
                               )
np.savez(f'{exp_name}/results', updates=np.array(updates),
                            gradients=np.array(gradients),
                            dm1s=np.array(dm1s),
                            gradients_pred=np.array(gradients_pred),
                            dms=np.array(dms),)
# Save the network weights
torch.save(network.state_dict(), f'{exp_name}/network_weights.pth')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12,4))

# Adicionado marcador 'o-' para visualizar pontos isolados
axs[0].plot(fwi_loss, 'o-')
axs[0].set_title('Data Loss')
axs[0].set_xlabel('Iteration')
axs[0].spines['right'].set_visible(False)
axs[0].spines['top'].set_visible(False)

axs[1].plot(ssim_list, 'o-')
axs[1].set_title('SSIM')
axs[1].set_xlabel('Iteration')
axs[1].spines['right'].set_visible(False)
axs[1].spines['top'].set_visible(False)

plt.savefig(f'{exp_name}/losses.png',  bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
show_one_iter_dm(grad.cpu(), dm1.cpu(), g.cpu(), dm.cpu(), model.detach().cpu(), lossn, iteration=FWI_iter,
                cmap='bwr', vmin=m_vmin, vmax=m_vmax, extent=(0, par['nx']*par['dx']*1000, par['nz']*par['dx']*1000, 0),)

In [ ]:
show_model(updates[0], cmap='jet', vmin=m_vmin, vmax=m_vmax, figsize=(12, 5), extent=(0, par['nx']*par['dx']*1000, par['nz']*par['dx']*1000, 0),
           title='First update')

In [ ]:
# 1. Definir limites físicos consistentes para ambos os plots
m_vmin = model_true.min()
m_vmax = model_true.max()

# 2. Criar a moldura lado a lado
fig, axs = plt.subplots(2, 1, figsize=(15, 10))

# Plot 1: Modelo Inicial (Suave)
im0 = axs[0].imshow(model_init, cmap='jet', vmin=m_vmin, vmax=m_vmax,
                    extent=(0, par['nx']*par['dx']*1000, par['nz']*par['dx']*1000, 0))
axs[0].set_title('Modelo Inicial (Smooth Model)', fontsize=14)
axs[0].set_ylabel('Profundidade (m)')
plt.colorbar(im0, ax=axs[0], label='Velocidade (m/s)', fraction=0.015, pad=0.04)

# Plot 2: Primeiro Passo (Após U-Net / Hessiana)
im1 = axs[1].imshow(updates[0], cmap='jet', vmin=m_vmin, vmax=m_vmax,
                    extent=(0, par['nx']*par['dx']*1000, par['nz']*par['dx']*1000, 0))
axs[1].set_title('Primeiro Passo (Update 0 - Com Aproximação da Hessiana)', fontsize=14)
axs[1].set_xlabel('Distância (m)')
axs[1].set_ylabel('Profundidade (m)')
plt.colorbar(im1, ax=axs[1], label='Velocidade (m/s)', fraction=0.015, pad=0.04)

plt.tight_layout()
plt.show()